# Otimização de hiperparâmetros de modelos preditivos

- **Repositório:** `machine-learning-portfolio`  
- **Tecnologias:** Scikit-Learn  
- **Arquivo Original:** `TDE03 - Otimizacao.ipynb`  

---


## Import das Bibliotecas

In [2]:
import pandas as pd
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import numpy as np
import time


## Carregamento dos dados / Separação de features

In [3]:
train_data = pd.read_csv("mnist_train.csv", sep=",")
test_data = pd.read_csv("mnist_test.csv", sep=",")

# Separa as features e os rótulos do conjunto de treinamento
X_train = train_data.iloc[:, 1:]
y_train = train_data.iloc[:, 0].values

# .... conjunto de teste
X_test = test_data.iloc[:, 1:]
y_test = test_data.iloc[:, 0].values


## Baseline com Árvore de Decisão

In [4]:
#### BASELINE ####
# Árvore de decisão considerando todas as 784 features

model_baseline = tree.DecisionTreeClassifier(random_state=33)

start_time = time.time()
# Treinamento do modelo
model_baseline.fit(X_train, y_train)
end_time = time.time()

# Predições
y_pred_baseline = model_baseline.predict(X_test)
training_time_baseline = end_time - start_time

# Cálculo da acurácia
acc_baseline = accuracy_score(y_pred_baseline, y_test)

print(f"Acurácia do Baseline: {acc_baseline*100:.2f}%")
print(f"Tempo de treinamento do Baseline: {training_time_baseline:.2f} segundos")


Acurácia do Baseline: 87.71%
Tempo de treinamento do Baseline: 23.74 segundos


## Algoritmo Genético

In [7]:
def generate_initial_population(num_chromosomes, num_genes, prob):
    population = np.zeros((num_chromosomes, num_genes))
    num_features = max(1, int(num_genes * prob))
    for i in range(num_chromosomes):
        candidate = np.zeros(num_genes)
        random_genes = np.random.choice(range(num_genes), num_features, replace=False)
        candidate[random_genes] = 1
        population[i] = candidate
    return population

def eval_population(population, X_train_data, y_train_data, alpha=0.9):
    X_train_eval, X_val_eval, y_train_eval, y_val_eval = train_test_split(
        X_train_data, y_train_data, test_size=0.3, random_state=42)

    fitness_scores = []
    for i in range(len(population)):
        features_selected = np.where(population[i] == 1)[0]
        if len(features_selected) == 0:
            fitness_scores.append(0)
            continue
        model = tree.DecisionTreeClassifier(random_state=33)
        model.fit(X_train_eval.iloc[:, features_selected], y_train_eval)
        predictions = model.predict(X_val_eval.iloc[:, features_selected])
        acc = accuracy_score(y_val_eval, predictions)
        feature_penalty = len(features_selected) / num_genes
        fitness = alpha * acc + (1 - alpha) * (1 - feature_penalty)
        fitness_scores.append(fitness)
    return fitness_scores

def tournament_selection(population, fitness_scores, num_parents, tournament_size=3):
    selected_parents = []
    for _ in range(num_parents):
        participants_idx = np.random.choice(len(population), tournament_size, replace=False)
        participants_fitness = [fitness_scores[i] for i in participants_idx]
        winner_idx = participants_idx[np.argmax(participants_fitness)]
        selected_parents.append(population[winner_idx])
    return np.array(selected_parents)

def one_point_crossover(parents):
    offspring = []
    num_parents = len(parents)
    if num_parents % 2 != 0:
        parents = np.append(parents, [parents[0]], axis=0)
    for i in range(0, num_parents, 2):
        parent1 = parents[i]
        parent2 = parents[i+1]
        crossover_point = np.random.randint(1, len(parent1)-1)
        child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
        child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
        offspring.extend([child1, child2])
    return np.array(offspring)

def mutation(offspring, mutation_rate=0.02):
    for i in range(len(offspring)):
        for gene_index in range(len(offspring[i])):
            if np.random.rand() < mutation_rate:
                offspring[i][gene_index] = 1 - offspring[i][gene_index]
    return offspring

# Função para criar a nova população com elitismo
def create_new_population(population, offspring, fitness_scores, num_elites=2):
    sorted_indices = np.argsort(fitness_scores)[::-1]
    elites = population[sorted_indices[:num_elites]]
    num_offspring_needed = len(population) - num_elites
    new_population = np.vstack((elites, offspring[:num_offspring_needed]))
    return new_population


In [8]:
num_generations = 20
population_size = 30
num_genes = X_train.shape[1]
prob = 0.05
num_parents = population_size // 2
mutation_rate = 0.03
num_elites = 2
alpha = 0.9

start_search_time = time.time()
population = generate_initial_population(population_size, num_genes, prob)

for generation in range(num_generations):
    print(f"Geração {generation+1}")
    fitness_scores = eval_population(population, X_train, y_train, alpha)
    print(f"Melhor fitness nesta geração: {max(fitness_scores):.4f}")
    parents = tournament_selection(population, fitness_scores, num_parents)
    offspring = one_point_crossover(parents)
    offspring = mutation(offspring, mutation_rate)
    population = create_new_population(population, offspring, fitness_scores, num_elites)

end_search_time = time.time()
search_time = end_search_time - start_search_time
print(f"Tempo total para encontrar o subconjunto de features: {search_time:.2f} segundos")

# Avaliação do melhor cromossomo no conjunto de teste
best_fitness_idx = np.argmax(fitness_scores)
best_chromosome = population[best_fitness_idx]
selected_features = np.where(best_chromosome == 1)[0]

start_training_time = time.time()
model_ag = tree.DecisionTreeClassifier(random_state=33)
model_ag.fit(X_train.iloc[:, selected_features], y_train)
end_training_time = time.time()
training_time_ag = end_training_time - start_training_time

y_pred_ag = model_ag.predict(X_test.iloc[:, selected_features])
acc_ag = accuracy_score(y_test, y_pred_ag)

print(f"Acurácia com AG: {acc_ag*100:.2f}%")
print(f"Tempo de treinamento com AG: {training_time_ag:.2f} segundos")
print(f"Porcentagem de features selecionadas: {len(selected_features)/num_genes*100:.2f}%")



Geração 1
Melhor fitness nesta geração: 0.7426
Geração 2
Melhor fitness nesta geração: 0.8119
Geração 3
Melhor fitness nesta geração: 0.8213
Geração 4
Melhor fitness nesta geração: 0.8340
Geração 5
Melhor fitness nesta geração: 0.8400
Geração 6
Melhor fitness nesta geração: 0.8409
Geração 7
Melhor fitness nesta geração: 0.8466
Geração 8
Melhor fitness nesta geração: 0.8466
Geração 9
Melhor fitness nesta geração: 0.8466
Geração 10
Melhor fitness nesta geração: 0.8476
Geração 11
Melhor fitness nesta geração: 0.8496
Geração 12
Melhor fitness nesta geração: 0.8496
Geração 13
Melhor fitness nesta geração: 0.8496
Geração 14
Melhor fitness nesta geração: 0.8496
Geração 15
Melhor fitness nesta geração: 0.8496
Geração 16
Melhor fitness nesta geração: 0.8496
Geração 17
Melhor fitness nesta geração: 0.8496
Geração 18
Melhor fitness nesta geração: 0.8496
Geração 19
Melhor fitness nesta geração: 0.8501
Geração 20
Melhor fitness nesta geração: 0.8501
Tempo total para encontrar o subconjunto de featu

## Wrapper

In [9]:
import numpy as np
import time
from sklearn import tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

In [10]:
# Escalonamento dos dados para o intervalo [0, 1]
X_train = X_train / 255.0
X_test = X_test / 255.0

# Definindo X_train_reduced e X_test_reduced se necessário
X_train_reduced = X_train
X_test_reduced = X_test

In [22]:
def forward_selection(X_train_data, y_train_data, max_features):
    X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
        X_train_data, y_train_data, test_size=0.3, random_state=42)

    selected_features = []
    remaining_features = list(range(X_train_data.shape[1]))
    best_score = 0
    feature_count = 0
    start_time = time.time()

    while feature_count < max_features and remaining_features:
        scores = []
        for feature in remaining_features:
            # Testa a adição da próxima feature
            features_to_test = selected_features + [feature]
            # Treina o modelo com as features selecionadas
            model = tree.DecisionTreeClassifier(random_state=33)
            model.fit(X_train_fs.iloc[:, features_to_test], y_train_fs)
            # Avalia no conjunto de validação
            predictions = model.predict(X_val_fs.iloc[:, features_to_test])
            acc = accuracy_score(y_val_fs, predictions)
            scores.append((acc, feature))
        # Seleciona a feature que mais melhora o desempenho
        scores.sort(reverse=True)
        best_acc, best_feature = scores[0]
        if best_acc > best_score:
            best_score = best_acc
            selected_features.append(best_feature)
            remaining_features.remove(best_feature)
            feature_count += 1
            print(f"Feature adicionada: {best_feature}, Acurácia: {best_acc:.4f}")
        else:
            # Critério de parada: não há melhoria na acurácia
            print("Nenhuma melhoria na acurácia. Parando a seleção.")
            break
    end_time = time.time()
    search_time = end_time - start_time
    return selected_features, best_score, search_time

# Define o número máximo de features a serem selecionadas
max_features = 15
# max_features = int(X_train.shape[1] * 0.3)

start_search_time = time.time()

# Executa o algoritmo de Forward Selection
selected_features_fs, best_score_fs, search_time_fs = forward_selection(X_train, y_train, max_features)

end_search_time = time.time()
total_search_time_fs = end_search_time - start_search_time

print(f"Tempo total para encontrar o subconjunto de features: {total_search_time_fs:.2f} segundos")
print(f"Número de features selecionadas: {len(selected_features_fs)}")

# Treinamento final com o conjunto completo de treinamento e features selecionadas
start_training_time_fs = time.time()
model_fs = tree.DecisionTreeClassifier(random_state=33)
model_fs.fit(X_train.iloc[:, selected_features_fs], y_train)
end_training_time_fs = time.time()
training_time_fs = end_training_time_fs - start_training_time_fs

# Avaliação no conjunto de teste
y_pred_fs = model_fs.predict(X_test.iloc[:, selected_features_fs])
acc_fs = accuracy_score(y_test, y_pred_fs)

# Impressão dos resultados
print(f"Acurácia com Forward Selection: {acc_fs*100:.2f}%")
print(f"Tempo de treinamento: {training_time_fs:.2f} segundos")
print(f"Porcentagem de features selecionadas: {len(selected_features_fs)/X_train.shape[1]*100:.2f}%")


Feature adicionada: 406, Acurácia: 0.2444
Feature adicionada: 409, Acurácia: 0.3447
Feature adicionada: 542, Acurácia: 0.3914
Feature adicionada: 596, Acurácia: 0.4391
Feature adicionada: 459, Acurácia: 0.4829
Feature adicionada: 377, Acurácia: 0.5263
Feature adicionada: 318, Acurácia: 0.5631
Feature adicionada: 211, Acurácia: 0.6086
Feature adicionada: 298, Acurácia: 0.6500
Feature adicionada: 154, Acurácia: 0.6878
Feature adicionada: 428, Acurácia: 0.7198
Feature adicionada: 295, Acurácia: 0.7467
Feature adicionada: 517, Acurácia: 0.7663
Feature adicionada: 208, Acurácia: 0.7785
Feature adicionada: 655, Acurácia: 0.7900
Tempo total para encontrar o subconjunto de features: 3003.68 segundos
Número de features selecionadas: 15
Acurácia com Forward Selection: 79.19%
Tempo de treinamento: 0.67 segundos
Porcentagem de features selecionadas: 1.91%
